# Lesson 1.8: EDA Basic

Welcome to Exploratory Data Analysis. This notebook takes one raw, messy business file and walks the
full path: understanding its structure, cleaning it, transforming it, and moving it in and out of
files.

**Structure — the four learning outcomes, in order:**
* **Part 1: Descriptive Statistics** — *summarise* a dataset: shape, data types, distributions.
* **Part 2: Data Quality** — *handle* the messy reality: missing values, duplicates, impossible values.
* **Part 3: Data Transformation** — *transform* for analysis: mapping, labels, strings, categories, dates.
* **Part 4: Reading & Writing Data** — *read and write* CSV, JSON, Excel, databases.

**For Learners:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 150 minutes.** One messy file, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Descriptive Statistics | **Summarise** a dataset: shape, dtypes, distributions | 33 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Quality | **Handle** missing values, duplicates, impossible values | 42 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Data Transformation | **Transform**: mapping, labels, strings, categories, dates, grouping | 35 min |
> | **Part 4** | Reading & Writing Data | **Read and write** CSV, JSON, Excel, databases | 15 min |
>
> **The spine:** we work on one file, `data/cafe_june_raw.csv`, from start to finish. Each section
> improves the same `clean` table, and Part 4 saves it. Small hand-built tables appear alongside it
> as *drills* — they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`;
> the Appendix at the end is self-study.


### The business problem

> **The Daily Grind** is a four-outlet café chain in Singapore. Revenue has been flat for two
> quarters, and the owner has to decide whether to renew the Marina Bay lease. She asks her
> assistant to send you the sales data. What arrives is a **raw till export**: one row per outlet,
> per day, per part of the day, straight out of the point-of-sale system, untouched.
>
> Nobody can answer the owner's question from this file yet. Today's job is to make it
> answerable — and to be able to say *why* every number in it can be trusted.

This is the first of three lessons on the same problem:

| Lesson | The question | What you do |
|---|---|---|
| **1.8 — today** | **Can I trust this data?** | clean one month of the raw export |
| 1.9 | What is the pattern? | 18 months, cleaned: time, joins, grouping |
| 1.10 | How do I make them act? | one chart, one slide, one decision |


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Load the two toolkits we need. `pd` and `np` are just short nicknames so we can
#    type `pd.something` instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np


In [ ]:
# 👉 Load the dataset we will use all session: June 2025's till export from four cafés.
#    `read_csv` reads a comma-separated text file into a DataFrame -- a table with named columns.
#    pandas already treats an empty field, "NA" and "n/a" as missing.
raw = pd.read_csv("../data/cafe_june_raw.csv")

raw


### 🎬 Why this matters — before you trust a single number

Run the next three cells. The chain has **four** cafés and its busiest shift takes about \$1,000.


In [ ]:
# 👉 `.value_counts()` counts how many rows have each value. How many cafés do you count?
raw["outlet"].value_counts()


In [ ]:
# 👉 The same question of the daypart column. There are three parts to a trading day.
raw["daypart"].value_counts()


In [ ]:
# 👉 Sort the takings column and look at the two ends. `.dropna()` skips the blank cells,
#    because a sort cannot compare text with a blank -- which is itself a clue.
#    `.iloc[[0, -1]]` takes the first and last rows of the sorted result.
raw["revenue_raw"].dropna().sort_values().iloc[[0, -1]]


**Three problems, in three lines of output.**

1. **Twelve spellings for four cafés.** `Raffles Place`, `raffles place`, `RAFFLES PLACE`,
   `Raffles Pl.`… Group by outlet today and you get twelve cafés, four of which are the same shop.
2. **Nine labels for three dayparts** — `Morning`, `morning`, `AM`, and so on.
3. **The revenue column is not a number.** Sorted, the "smallest" value is `" 1,006.71 "` and the
   "largest" is `"S$94.41"`, because pandas is comparing them as **text**: a space sorts before a
   digit, and the letter `S` sorts after every digit. Sorted as text, \$98,000 loses to \$99.

Any average, chart or model built on this file is wrong before you start. Worse, none of it would
*look* wrong: it would produce numbers, with decimal places, and nobody in the meeting would know.

Part 1 is the routine that finds problems like these in about two minutes.


---

## Part 1: Descriptive Statistics — Summarising Your Data

**Learning outcome 1:** *Summarise a dataset using descriptive statistics and identify its shape,
data types, and distributions.*

**Goal:** know what you are holding before you touch it.

⏱️ ~33 min including Group Exercise 1


### 1.1: The First Look — a five-move first look

Do these five, in this order, every time you meet a new file. It takes two minutes and it is the
difference between analysis and guesswork.

| Move | Question it answers |
|---|---|
| `.head()` | What do the rows actually look like? |
| `.shape` | How big is it? |
| `.info()` | What type is each column, and where are the holes? |
| `.dtypes` | Is anything stored as the wrong type? |
| `.describe()` | Are the numbers plausible? |


In [ ]:
# 👉 Move 1: look at real rows. `.head()` shows the first 5 (pass a number for more).
#    Never skip this. Half of all data problems are visible to the naked eye.
raw.head()


In [ ]:
# 👉 Move 2: how big? `.shape` gives (rows, columns). No brackets -- it is a value, not a method.
raw.shape


In [ ]:
# 👉 Move 3: the single most useful command in pandas. For every column it reports the
#    non-null count and the type. Compare each count with the row count above: the gaps are holes.
raw.info()


In [ ]:
# 👉 Move 4: just the types. `object` means text (or mixed). Note `revenue_raw` and `date_text`
#    are text, not numbers and dates -- and `tickets` is a decimal, which is odd for a count.
raw.dtypes


In [ ]:
# 👉 Move 5: the numbers. Read the min and max rows first: that is where impossible values hide.
#    Notice which columns are MISSING from this table -- `.describe()` only sees numeric ones.
raw.describe()


In [ ]:
# 👉 `.describe()` skips text columns by default. Ask for them explicitly and you get
#    count / unique / top / freq -- which is where spelling variants show up.
raw.describe(include="object")


**Write down what the first look found** — this is our to-do list for Part 2:

1. **Missing values** — `revenue_raw` 3, `tickets` 2, `items` 3, `staff_on_shift` 5,
   `manager_email` 2, `notes` 332.
2. **Duplicate rows** — 366 rows, but June has only 30 dates (`date_text` shows 30 unique values),
   and 30 days × 4 cafés × 3 dayparts = **360** possible shifts. Six rows too many: something was
   sent twice.
3. **Wrong types** — revenue is text; the date is text; counts are decimals.
4. **Impossible values** — `tickets` has a minimum of **-4**, and a shift with **0** tickets that
   still took money.
5. **Twelve outlet spellings and nine daypart labels** for four cafés and three dayparts.

Five problems, five different right answers. That is Part 2 and Part 3.

> **Why `tickets` is a decimal.** A column of whole numbers with even one missing value cannot stay
> an integer, because there is no integer that means "missing". pandas quietly promotes the whole
> column to `float64`. A count stored as a decimal is therefore a *symptom*: it usually means the
> column has holes in it.


### 1.2a: Those Statistics, Unpacked on Our Data

`.describe()` is a bundle of simpler methods. Here we take them one at a time — on a slice of
`raw` small enough that you can check the arithmetic by hand.


In [ ]:
# 👉 `.loc[[...], [...]]` takes the listed rows and just two numeric columns. Small on purpose:
#    8 rows you can add up yourself. (`revenue_raw` cannot appear here -- it is still text.)
#    Row 12 is in the list deliberately: its ticket count is missing.
peek = raw.loc[[0, 1, 2, 3, 4, 5, 12, 13], ["tickets", "items"]]

peek


**Reductions** — a whole column in, one number out. By default they work *down* the rows.

Note the hole in row 12 of `tickets`: watch what each method does with it.


In [ ]:
# 👉 Total tickets sold and total items bought, across these 8 shifts.
#    One number per column comes back.
peek.sum()


In [ ]:
# 👉 The averages. `tickets` has 7 values, not 8, so `.mean()` divides by 7 -- it divides by
#    the number of values *present*. That is either exactly what you want or a silent bug,
#    depending on why the value is missing.
peek.mean()


**`skipna` — what to do about the hole.** By default pandas ignores missing values and
carries on. That is convenient, and it is also how a hole becomes invisible.


In [ ]:
# 👉 `skipna=False` refuses to silently work around the hole. `tickets` becomes NaN;
#    `items` still has a real total. Now you can *see* which column had a gap.
peek.sum(skipna=False)


**Indirect statistics:** finding *where* the max or min value sits (the row label).


In [ ]:
# 👉 Not the biggest *value* -- the row *label* where it sits. Combine it with `.loc[...]`
#    to pull the whole row out and look at the shift that did it.
peek["items"].idxmax()


In [ ]:
# 👉 Same for the minimum, on both columns at once.
peek.idxmin()


**Accumulations:** running totals.


In [ ]:
# 👉 A running total: each row is itself plus everything above it. Read the bottom row of
#    `items` -- that is the same number `.sum()` gave you.
peek.cumsum()


**`.describe()` — the bundle.** Everything above, in one call. You met it as move 5 of the first look.


In [ ]:
# 👉 All of the above in one call, plus the quartiles. This is why `.describe()` is move 5
#    and not five separate commands.
peek.describe()


**Categorical / non-numeric data:** `.describe()` behaves differently for text, showing counts
and the most frequent value rather than means.


In [ ]:
# 👉 A Series is a single column of data. This one holds text, so `.describe()` switches to
#    count / unique / top / freq. `top` is the most common value; `freq` is how often it appears.
raw["outlet"].describe()


In [ ]:
# 👉 List the distinct values, in the order they first appear. Duplicates are dropped.
raw["outlet"].unique()


In [ ]:
# 👉 Count how many times each distinct value appears, most frequent first.
raw["outlet"].value_counts()


### 1.2b: The `axis` Sandbox — a deliberately meaningless table

One piece of notation left: **`axis`**. Every reduction can run *down* the rows (the default) or
*across* the columns. This drill uses a made-up table, because on real data one of the two
directions is usually nonsense — and it is easier to see the mechanics when nothing is at stake.


In [ ]:
# 👉 Build a small table by hand. The outer [ ] is a list of rows; each inner [ ] is one row.
#    `np.nan` is how you write a missing value.
demo = pd.DataFrame(
    [[1.4, np.nan], [7.1, -4.5], [np.nan, np.nan], [0.75, -1.3]],
    index=["a", "b", "c", "d"],
    columns=["one", "two"],
)

demo


In [ ]:
# 👉 Add up each column, top to bottom. One number comes back per column.
demo.sum()


In [ ]:
# 👉 Same addition, but sideways: add across each row. `axis="columns"` means 'go across'.
demo.sum(axis="columns")


In [ ]:
# 👉 `skipna=False` says 'do NOT ignore missing values'. Any NaN in a column poisons its total.
demo.sum(skipna=False)


In [ ]:
# 👉 Same idea going across the rows instead of down the columns.
demo.sum(axis=1, skipna=False)


**Back to reality.** On our café export, `axis="columns"` is almost always wrong —
`tickets + staff_on_shift` is not a quantity that exists. Adding *down* a column
("total tickets in June") is the meaningful direction. Know both; reach for the first.


### 🛠️ Group Exercise 1 — Summarising (8 min)

The scaffold fades: task (a) is done for you, (b) is half-written, (c) and (d) are yours.

> **(a) Worked for you** — the cell below sorts a count table by its label instead of by count.
> Read it, then run it.
>
> **(b) Fill in the blanks** — sort the outlet counts by frequency, *smallest first*:
> ```python
> raw["outlet"].value_counts().sort_______(ascending=____)
> ```
> *Expected:* 12 rows, smallest count at the top.
>
> **(c) From scratch** — which shift sold the most items? Use `.idxmax()` on `raw["items"]` to get
> the row label, then `.loc[...]` to pull that whole row out.
> *Expected:* one row — the morning of Tuesday 10 June at Raffles Place, with 336 items.
>
> **(d) Explain, no code** — `raw.info()` reports 364 non-null values for `tickets` but the table
> has 366 rows. Where did the other 2 go, and why is showing them as missing better than showing 0?


**(a) Worked example** — count, then sort by label:


In [ ]:
# 👉 `.sort_index()` re-sorts that count table by the label (alphabetically) instead of by count.
#    Reading right to left: count the values, then sort the result.
raw["daypart"].value_counts().sort_index()


---

# ☕ Break — 10 minutes

**Where we are:** you can now describe *what* a dataset looks like.
**Next up:** Part 2 — fixing what is wrong with it (missing values, duplicates, impossible values).


---

## Part 2: Data Quality — Missing Data, Duplicates & Impossible Values

**Learning outcome 2:** *Handle missing values, duplicates, and outliers using appropriate Pandas
methods.*

**Goal:** work through the five problems on the Part 1 to-do list. Every fix follows the same four
beats — **find it → decide → apply → verify** — and that habit is worth more than any single method.

⏱️ ~42 min including Group Exercise 2


### 2.0: One type fix first

We cannot check whether the takings are plausible while they are **text**. This is the one line of
string-cleaning we need up front; the full toolkit is section 3.3.


In [ ]:
# 👉 `.copy()` makes an independent table, so `raw` stays as the untouched original --
#    which is what lets us compare "before and after" at the end of the session.
clean = raw.copy()

# 👉 Strip out everything that is not a digit, a dot or a minus sign: "S$1,240.50" -> "1240.50".
#    `regex=True` says "the thing I am searching for is a pattern, not literal text".
#    `to_numeric` then turns that text into a real number; `errors="coerce"` puts NaN
#    wherever the text could not be read as a number at all.
clean["revenue_sgd"] = pd.to_numeric(
    clean["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
)

clean[["revenue_raw", "revenue_sgd"]].head()


In [ ]:
# 👉 NOW `.describe()` can see the money column. Read the min and the max.
clean["revenue_sgd"].describe()


> **There it is.** A minimum of **-999** and a maximum of **98,000**, in a business whose busiest
> shift takes about \$1,000. Neither is a real number:
>
> - **-999** is a *sentinel* — the old till wrote it when a shift failed to close off. It means
>   "no reading", not "minus nine hundred and ninety-nine dollars".
> - **98,000** is a keying error: someone typed `98000` for a shift that took about \$980.
>
> Both were completely invisible while the column was text. This is why move 4 of the first look
> (`.dtypes`) exists.


### 2.1: Handling Missing Data

Missing data shows up as `NaN` (Not a Number) or `None`.


**Step 1 — find the holes on our own dataset.**


In [ ]:
# 👉 `.isna()` marks every cell True/False for "is this missing?". Chaining `.sum()`
#    counts the Trues per column, because True counts as 1.
clean.isna().sum()


In [ ]:
# 👉 But look at `notes` more closely. Some cells say "N.A." or "-" -- which a human reads as
#    empty and pandas reads as ordinary text. Fake blanks are worse than real ones: they are
#    invisible to `.isna()`.
clean["notes"].value_counts(dropna=False)


In [ ]:
# 👉 Turn the fake blanks into real ones, so `.isna()` tells the truth from here on.
clean["notes"] = clean["notes"].replace(["N.A.", "-"], np.nan)

print("missing notes before: 332")
print("missing notes after: ", clean["notes"].isna().sum())


**Step 2 — decide, column by column.** There is no single right answer, only defensible ones:

| Column | Holes | Decision | Why |
|---|---|---|---|
| `notes` | 362 | leave as `NaN` | "no note" is genuinely no information; do not invent one |
| `manager_email` | 2 | leave as `NaN` | you cannot guess an email address |
| `staff_on_shift` | 5 | fill with the **median** | staffing is fairly consistent; a typical value is a fair guess |
| `items` | 3 | fill with the **median** | same reasoning |
| `revenue_sgd` | 3 | **wait** | the sentinels must go first, or the median is poisoned |
| `tickets` | 2 | **wait** | there are impossible values in here too |

The last two rows are the lesson. **Order matters**, and section 2.3 is where it bites.


In [ ]:
# 👉 Apply the two easy decisions. A dictionary lets you use a different filler per column.
#    `.median()` is the middle value, which ignores lopsided extremes -- safer than the mean.
clean = clean.fillna({
    "staff_on_shift": clean["staff_on_shift"].median(),
    "items": clean["items"].median(),
})

clean[["staff_on_shift", "items"]].isna().sum()


Drills on smaller data follow, so you can see each method in isolation.


In [ ]:
# 👉 A Series of numbers where one entry is missing.
float_data = pd.Series([1.2, -3.5, np.nan, 0])

float_data


In [ ]:
# 👉 `.isna()` answers 'is this one missing?' for every entry: True means missing.
float_data.isna()


The built-in Python `None` value is also treated as missing in pandas object columns.


In [ ]:
# 👉 Python's own `None` also counts as missing, alongside `np.nan`.
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])

string_data


In [ ]:
# 👉 Proof: both `np.nan` and `None` come back as True.
string_data.isna()


### Strategy 1: Dropping Missing Data (`dropna`)

The simplest strategy is to remove the rows or columns that contain missing values.


In [ ]:
# 👉 `.dropna()` returns a copy with the missing entries removed. The original is unchanged --
#    nothing in pandas edits in place unless you assign the result back.
float_data.dropna()


In [ ]:
# 👉 The manual version of the same thing: `notna()` marks the good rows, and putting that
#    True/False mask in square brackets keeps only the True ones.
float_data[float_data.notna()]


With DataFrames, `dropna` by default drops **any row** containing **any** missing value.


In [ ]:
# 👉 A 4-row table with missing values scattered around.
demo_df = pd.DataFrame([[1., 6.5, 3.], [1., np.nan, np.nan],
                        [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])

demo_df


In [ ]:
# 👉 Default behaviour: drop a row if it has *any* missing value. Harsh -- only row 0 survives.
demo_df.dropna()


`how="all"` only drops rows where **every** value is missing.


In [ ]:
# 👉 Gentler: only drop a row where every single value is missing.
demo_df.dropna(how="all")


To drop **columns** instead of rows, pass `axis="columns"`.


In [ ]:
# 👉 Add a brand-new column that is entirely missing, so we have something to drop.
demo_df[4] = np.nan

demo_df


In [ ]:
# 👉 `axis="columns"` switches the target from rows to columns, so the all-missing column goes.
demo_df.dropna(axis="columns", how="all")


You can also set a **threshold**: keep only rows with at least `n` real values.


In [ ]:
# 👉 A 7-row, 3-column table of random numbers, then poke holes in it.
#    `.iloc[rows, columns]` selects by position, so :4 means "the first four rows".
rng = np.random.default_rng(seed=12345)
holes = pd.DataFrame(rng.standard_normal((7, 3)))
holes.iloc[:4, 1] = np.nan
holes.iloc[:2, 2] = np.nan

holes


In [ ]:
# 👉 With the default settings, most rows are gone.
holes.dropna()


In [ ]:
# 👉 `thresh=2` means 'keep the row if it has at least 2 real (non-missing) values'.
holes.dropna(thresh=2)


### Strategy 2: Filling Missing Data (`fillna`)

Instead of losing data, fill the holes with a constant or a calculated value.


In [ ]:
# 👉 `.fillna()` plugs every hole with a value instead of deleting the row.
holes.fillna(0)


You can specify a different fill value for each column:


In [ ]:
# 👉 Pass a dictionary to use a different filler per column: {column_name: fill_value}.
holes.fillna({1: 0.5, 2: 0})


**Forward / backward fill:** carry a neighbouring value into the gap. Common with time series.


In [ ]:
# 👉 `bfill` = backward fill: copy the next real value upwards into the gap.
holes.bfill()


In [ ]:
# 👉 `limit=2` stops the copying after 2 rows, so long gaps are not silently invented.
holes.bfill(limit=2)


**Imputation:** filling with the mean or median is a very common technique.


In [ ]:
# 👉 A small Series with two holes in it.
s_holes = pd.Series([1., np.nan, 3.5, np.nan, 7])

s_holes


In [ ]:
# 👉 'Imputation': fill the holes with the average of the values we do have.
#    `s_holes.mean()` is computed from the 3 real values only.
s_holes.fillna(s_holes.mean())


### 2.2: Handling Duplicates

Duplicate rows inflate every total built from them. The first look said 366 rows for a month that can
only contain 360 outlet-day-daypart combinations.


In [ ]:
# 👉 `.duplicated()` flags a row True if an identical row appeared earlier.
#    `.sum()` counts them.
clean.duplicated().sum()


In [ ]:
# 👉 Show the offending rows so you can eyeball them before deleting anything.
#    `keep=False` marks *both* copies, not just the later one, so they sit together.
clean[clean.duplicated(keep=False)].sort_values(["date_text", "outlet", "daypart"]).head(8)


In [ ]:
# 👉 Genuine full-row copies, so drop them. Assign back for the change to stick.
clean = clean.drop_duplicates()

clean.shape


360 rows. **Note the safer habit:** here the whole row was identical, so `drop_duplicates()` is
enough. In real exports the same shift is often sent twice with a *corrected* figure, and then the
rows are not identical — you have to say which columns identify a shift, and which copy to keep.

Check it explicitly:


In [ ]:
# 👉 A shift is identified by outlet + date + daypart. After de-duplicating there should be
#    exactly one row per combination -- but the outlet column is still spelled 12 ways,
#    so this check cannot pass until Part 3 fixes that. Try it and see.
clean.duplicated(subset=["date_text", "outlet", "daypart"]).sum()


> **Read that number carefully.** It is 0 — and that is *not* reassuring. Every "Marina Bay" row
> and every "marina bay" row look like different outlets to pandas, so a genuine double-submission
> under two spellings would slip straight through this check. **You cannot de-duplicate reliably
> until the key columns are standardised**, which is Part 3. Note it down and come back.


In [ ]:
# 👉 Build a table from a dictionary: each key becomes a column name, each list becomes a column.
dupes = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"], "k2": [1, 1, 2, 3, 3, 4, 4]})

dupes


In [ ]:
# 👉 For each row: 'have I already seen this exact row above?' True means it is a repeat.
dupes.duplicated()


In [ ]:
# 👉 Keep only the first appearance of each row and throw the repeats away.
dupes.drop_duplicates()


**Subset:** sometimes you only care about duplicates in specific columns.


In [ ]:
# 👉 Add a column of running numbers so we can see which rows survive the next steps.
dupes["v1"] = range(7)

dupes


In [ ]:
# 👉 `subset=` narrows the comparison: rows count as duplicates if column k1 matches,
#    even where v1 differs. This is the version you want for "same shift, sent twice".
dupes.drop_duplicates(subset=["k1"])


**Keep:** by default pandas keeps the first occurrence. Keep the last instead with `keep="last"`.


In [ ]:
# 👉 With a corrected re-submission, the LAST copy is usually the one you want.
dupes.drop_duplicates(subset=["k1", "k2"], keep="last")


### 2.3: Handling Impossible Values and Outliers

An **outlier** is a value far from the rest. Some are real and important; some are errors. The
question is never "is it extreme?" but **"is it possible?"**


**On our dataset:** four impossible-value problems, and they need different treatments.


In [ ]:
# 👉 Boolean filtering: build a True/False test, put it in the square brackets, and only
#    the True rows come back. `|` means OR.
impossible = clean[
    (clean["revenue_sgd"] < 0) | (clean["revenue_sgd"] > 20000) | (clean["tickets"] <= 0)
]

impossible[["outlet", "date_text", "daypart", "revenue_sgd", "tickets"]]


Eight rows, four different problems, four different right answers:

| What you see | What it means | Decision |
|---|---|---|
| `revenue_sgd` = **-999** (×4) | sentinel: the till failed to close off | not a number at all → `NaN`, then fill |
| `revenue_sgd` = **98,000** | keyed `98000` for about `980.00` | we cannot recover it → `NaN`, then fill |
| `tickets` = **0** with revenue > 0 | money taken, no receipts counted | impossible combination → `NaN`, then fill |
| `tickets` = **-4** | a negative count of customers | impossible → `NaN`, then fill |

Notice that all four end in `NaN`. **That is deliberate:** turning a wrong number into an explicit
hole is honest, and it hands the decision to the fill step where you have to state your reasoning.
Silently overwriting it with a plausible-looking number is how bad data survives.


In [ ]:
# 👉 `.mask(condition)` replaces values where the condition is True with NaN.
#    Read it as "hide the values I cannot believe".
clean["revenue_sgd"] = clean["revenue_sgd"].mask(
    (clean["revenue_sgd"] < 0) | (clean["revenue_sgd"] > 20000)
)
clean["tickets"] = clean["tickets"].mask(clean["tickets"] <= 0)

clean[["revenue_sgd", "tickets"]].describe()


Now the ranges are plausible — and we have created new holes on purpose. Eight shifts need a value:
the four sentinels, the mis-keyed \$98,000, and the three that were blank from the start.

**Fill them in the right order — sentinels out first, statistic second.** The next cell shows exactly
how much that order is worth, and the answer is more interesting than "a lot".


In [ ]:
# 👉 What the ordering is actually worth. Compare the statistic computed BEFORE masking
#    (with -999 and 98,000 still in the column) against the same statistic computed after.
#    (`raw.drop_duplicates()` first, so we are comparing like with like -- the duplicated
#     batch is already out of `clean`.)
before = pd.to_numeric(
    raw.drop_duplicates()["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True),
    errors="coerce",
)

print(f"median before masking: ${before.median():>8,.2f}   after: ${clean['revenue_sgd'].median():>8,.2f}")
print(f"mean   before masking: ${before.mean():>8,.2f}   after: ${clean['revenue_sgd'].mean():>8,.2f}")


> **Read those two lines carefully — they are the real lesson.**
>
> The **median** barely moves: \$456 → \$463. That is not luck; it is the whole reason a median is
> the safer default. It looks at the middle of the sorted values, so four absurd lows and one absurd
> high hardly shift it.
>
> The **mean** moves from \$742 to \$486. Had you filled with the mean in the wrong order, all eight
> filled shifts would have been **53% too high**, and the month's total would have been overstated by
> thousands.
>
> So the rule is not "the median gets poisoned" — it is: **mask first, then compute, because you
> cannot see from the outside which case you are in.** Today the median gave you a safety net. The
> next dataset might have thirty sentinels instead of four, and then even the median moves. Order the
> steps correctly and you never have to know.


In [ ]:
# 👉 Median, not mean, for exactly the reason above. (In Lesson 1.9 you will learn to fill per
#    outlet and daypart, which is finer and better; a single median is honest enough for now.)
clean = clean.fillna({
    "revenue_sgd": clean["revenue_sgd"].median(),
    "tickets": clean["tickets"].median(),
})

print("holes left in revenue_sgd:", clean["revenue_sgd"].isna().sum())
print("holes left in tickets:    ", clean["tickets"].isna().sum())


**Compare with the original to see what cleaning bought us:**


In [ ]:
# 👉 The same question asked of the dirty file and the clean one. This is the number that
#    would have gone into the owner's report.
dirty_total = pd.to_numeric(
    raw["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
).sum()

print(f"June revenue, raw file:   ${dirty_total:,.2f}")
print(f"June revenue, cleaned:    ${clean['revenue_sgd'].sum():,.2f}")
print(f"difference:               ${dirty_total - clean['revenue_sgd'].sum():,.2f}")


> **The raw file overstates June by more than \$90,000 — about 52%.** One mis-keyed shift did most
> of it, six duplicated rows added more, and four sentinels quietly pulled in the other direction.
>
> Every one of those was a two-line fix. None of them announced itself. A report built on the raw
> file would have told the owner her chain had its best month ever.

Drills on random data follow, where the mechanics are easier to see.


In [ ]:
# 👉 1000 rows of random numbers shaped like a bell curve, so most values sit near 0.
#    Values beyond about ±3 are rare by construction -- which makes them useful practice outliers.
spread = pd.DataFrame(np.random.default_rng(seed=12345).standard_normal((1000, 4)))

spread.describe()


**Detection:** find values more than 3 away from zero.


In [ ]:
# 👉 Grab one column, then keep only the entries more than 3 away from zero.
#    `.abs()` ignores the sign, so it catches both tails at once.
col = spread[2]

col[col.abs() > 3]


To find **any row** with an outlier in **any column**, use `.any(axis="columns")`.


In [ ]:
# 👉 Same test applied to the whole table. `.any(axis="columns")` asks, per row,
#    'was there at least one True?'
spread[(spread.abs() > 3).any(axis="columns")]


**Capping:** instead of removing outliers, pull them back to a threshold.


In [ ]:
# 👉 'Capping': pull extreme values back to the ±3 boundary instead of deleting the row.
#    `np.sign` keeps the direction (-1 or +1) so a low outlier becomes -3, not +3.
spread[spread.abs() > 3] = np.sign(spread) * 3

spread.describe()


**Removal:** or drop the rows entirely.


In [ ]:
# 👉 The other option -- 'trimming': keep only rows where every column is within bounds.
#    `~` flips True/False, so this reads "not (any column out of bounds)".
spread[~(spread.abs() > 2.9).any(axis="columns")].shape


### 🛠️ Group Exercise 2 — Data Quality (12 min)

Same fading. Work on `practice` (created below) — a small random table with holes punched in it,
so every change is visible.

> **(a) Worked for you** — run the cell below and read the comment.
>
> **(b) Fill in the blanks** — drop only the rows with *fewer than 2* real values:
> ```python
> practice.dropna(______=2)
> ```
> *Expected:* 5 of the 6 rows survive.
>
> **(c) From scratch** — fill the holes in column `b` with that column's median, leaving the other
> columns untouched. *Hint:* `.fillna({...})` takes a dictionary.
>
> **(d) On the real data, from scratch** — how many rows of `raw` had a `revenue_raw` value that
> could not be read as a number at all? *Hint:* re-run the `to_numeric(..., errors="coerce")` line
> on `raw` and count the NaNs. Then say, in one sentence, why `errors="coerce"` is safer than
> letting the conversion raise an error.


**(a)–(c)** use the `practice` table below.


In [ ]:
# 👉 Practice data for the exercise: 6 rows of random numbers with holes punched in three
#    places. `.iloc[row, column]` selects one cell by position.
practice = pd.DataFrame(
    np.random.default_rng(seed=7).standard_normal((6, 3)).round(2), columns=["a", "b", "c"]
)
practice.iloc[0, 1] = np.nan
practice.iloc[2, 1] = np.nan
practice.iloc[4, [0, 1, 2]] = np.nan

practice


> **Stuck?** Every method you need appeared in 2.1–2.3 above. The pattern is always: build the
> test → look at what it catches → apply the fix → verify with `.isna().sum()`.


---

# ☕ Break — 10 minutes

**Where we are:** the numbers are now plausible and the duplicates are gone.
**Next up:** Part 3 — the text columns, the dates, and the answer the owner actually asked for.


---

## Part 3: Data Transformation

**Learning outcome 3:** *Transform data through type conversion, string cleaning, and categorical
encoding.*

**Goal:** `clean` now has believable numbers, but you still cannot group by outlet (twelve
spellings), ask a date question (the date is text), or produce a per-outlet summary. That is this
section.

⏱️ ~35 min including Group Exercise 3


### 3.1: Transforming Data (Mapping)

A **mapping** is a lookup table: for each messy value, the value you want instead.


**On our dataset:** twelve outlet spellings for four cafés, and nine daypart labels for three
dayparts. A dictionary maps every messy spelling to the one we want.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 See the mess in full before writing the mapping. `.unique()` lists distinct values.
clean["outlet"].unique()


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 A dictionary is a lookup table: {what_is_in_the_data: what_we_want}.
#    Writing it out by hand is not inelegant -- it is a record of a business decision.
outlet_map = {
    "Raffles Place": "Raffles Place",
    "raffles place": "Raffles Place",
    "RAFFLES PLACE": "Raffles Place",
    "Raffles Pl.": "Raffles Place",
    "Tampines Mall": "Tampines Mall",
    "tampines mall": "Tampines Mall",
    "Tampines  Mall": "Tampines Mall",
    "Marina Bay": "Marina Bay",
    "marina bay": "Marina Bay",
    "Marina Bay ": "Marina Bay",
    "Holland Village": "Holland Village",
    "Holland V": "Holland Village",
}

clean["outlet_name"] = clean["outlet"].map(outlet_map)

clean["outlet_name"].value_counts()


Four cafés. **A warning worth internalising:** if a thirteenth spelling appears next month,
`.map()` turns it into `NaN` silently. Always check afterwards.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 The check that catches a spelling you did not know about.
#    If this is not 0, something in the outlet column was not in your dictionary.
clean["outlet_name"].isna().sum()


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 The daypart column has the same problem, and gets the same treatment.
daypart_map = {
    "Morning": "Morning", "morning": "Morning", "AM": "Morning",
    "Midday": "Midday", "midday": "Midday", "Lunch": "Midday",
    "Evening": "Evening", "evening": "Evening", "PM": "Evening",
}

clean["daypart"] = clean["daypart"].map(daypart_map)

clean["daypart"].value_counts()


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 And now the duplicate check from 2.2 can finally do its job: one row per outlet,
#    per date, per daypart. This is the check that was meaningless before the mapping.
clean.duplicated(subset=["date_text", "outlet_name", "daypart"]).sum()


Drills on smaller examples follow.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A tiny table of foods and weights, built from a dictionary of column-name: values.
foods = pd.DataFrame({
    "food": ["bacon", "pulled pork", "bacon", "pastrami", "corned beef", "bacon", "pastrami"],
    "ounces": [4, 3, 12, 6, 7.5, 8, 3],
})

foods


**Scenario:** add a column showing the animal each food came from.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A dictionary = a lookup table. Given a food (the key) it hands back an animal (the value).
meat_to_animal = {
    "bacon": "pig", "pulled pork": "pig", "pastrami": "cow", "corned beef": "cow",
}

meat_to_animal


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.map()` walks down the food column and swaps each value for its dictionary match.
#    Assign the result to a new column so the original stays intact.
foods["animal"] = foods["food"].map(meat_to_animal)

foods


You can also pass a **function** to `map()` for custom logic.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.map()` also accepts a function. `def` defines one: it takes an input x and returns
#    the looked-up value. Same result, more flexible.
def get_animal(x):
    return meat_to_animal[x]

foods["food"].map(get_animal)


**Replacing values:** `replace` is a specialised version of `map`, ideal for sentinel values.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Some systems use a fake number like -999 to mean 'nothing recorded' -- exactly like
#    the till in our own dataset.
sentinel_s = pd.Series([1., -999., 2., -999., -1000., 3.])

sentinel_s


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.replace()` swaps one value for another -- here, the fake code becomes a proper NaN.
sentinel_s.replace(-999, np.nan)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Pass a list to replace several values with the same thing in one go.
sentinel_s.replace([-999, -1000], np.nan)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Two lists of equal length: first list is what to find, second is what to put in its place.
sentinel_s.replace([-999, -1000], [np.nan, 0])


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 The clearest form: a dictionary of {old_value: new_value}. Same result, easier to read.
sentinel_s.replace({-999: np.nan, -1000: 0})


> **`.map()` vs `.replace()` — say which behaviour you want before you pick.**
> `.map()` needs an entry for *every* value and turns anything unlisted into `NaN`.
> `.replace()` changes only what you list and leaves everything else alone.
> For standardising a column with a known set of values, `.map()`'s strictness is a feature: it
> tells you when something new shows up.


### 3.2: Renaming Axis Labels

Changing row and column labels, using the same mapping idea.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A 3x4 table of the numbers 0-11. `np.arange(12)` makes 0..11 in a line and
#    `.reshape((3, 4))` folds it into 3 rows of 4.
labels = pd.DataFrame(
    np.arange(12).reshape((3, 4)),
    index=["Tampines", "Bishan", "Yishun"],
    columns=["one", "two", "three", "four"],
)

labels


Using `.map()` on the index:


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A function that shortens a label to its first 4 characters and upper-cases it.
#    `x[:4]` takes characters 0 to 3.
def shorten(x):
    return x[:4].upper()

labels.index.map(shorten)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.index.map()` applies that function to every row label. Assigning back to `labels.index`
#    makes the change stick.
labels.index = labels.index.map(shorten)

labels


Using `.rename()` (which returns a copy by default):


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.rename()` is the tidier way to relabel. `str.title` and `str.upper` are ready-made
#    functions, passed without brackets because we want the function itself, not its result.
labels.rename(index=str.title, columns=str.upper)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.rename()` also takes dictionaries when you only want to change specific labels.
labels.rename(index={"YISH": "NORTH"}, columns={"three": "peekaboo"})


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 On our own table: give the columns the names a report would use.
clean = clean.rename(columns={"staff_on_shift": "staff", "manager_email": "email"})

clean.columns


### 3.3: String Manipulation

Pandas has a special accessor `.str` that unlocks text methods for a whole column at once, and
handles missing values gracefully.


**On our dataset:** the `manager` column has stray spaces and inconsistent case, and we want the
domain out of each manager's email address.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 Find the problem first. `repr()` shows the invisible characters, so spaces become visible.
[repr(v) for v in clean["manager"].unique()]


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `.str` applies a text method to the whole column at once. Chain them left to right:
#    strip the spaces off the ENDS, squeeze any doubled spaces in the MIDDLE down to one,
#    then title-case what is left. s`\+` means "one or more whitespace characters".
clean["manager"] = (
    clean["manager"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
)

clean["manager"].unique()


**Why three methods and not two.** `.str.strip()` only removes spaces at the *ends*, so
`"Priya  Nair"` would have survived it — and one manager would have appeared twice in every summary,
with her shifts split between the two spellings. The `\s+` replacement is what catches the doubled
space inside the name. Always print `.unique()` after a text clean-up and count the values.

That chain would have fixed most of the outlet column too — but not `Raffles Pl.` or `Holland V`,
which are abbreviations rather than typos. **Text cleaning handles the mechanical mess; a mapping
handles the decisions.**


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `.str.split("@")` cuts each email in two at the @ sign; `.str[1]` takes the second
#    piece -- the domain. Python counts from 0, so 1 is the second item.
clean["email"].str.split("@").str[1].value_counts(dropna=False)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A drill on the same idea, small enough to see. Start from a dictionary of name: email,
#    then turn it into a Series so the names become the row labels.
emails = pd.Series({
    "Aisha": "aisha.rahman@dailygrind.sg",
    "Wei Ming": "weiming.tan@dailygrind.sg",
    "Priya": "priya.nair@dailygrind.sg",
    "Daniel": "daniel.lim@dailygrind.com.sg",
    "Unknown": np.nan,
})

emails


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.str` unlocks text methods for a whole column at once. Note the missing entry stays NaN
#    rather than raising an error -- which is exactly why `.str` exists.
emails.str.contains("dailygrind")


Note on data types: pandas has a dedicated text type (`string`) as well as the generic `object`.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Convert to the dedicated text type. Mostly the same, but missing values behave more
#    predictably: you get a proper <NA> rather than a float NaN inside a text column.
emails_str = emails.astype("string")

emails_str


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Same test on the text-typed column: now the answer is a proper True/False/<NA>.
emails_str.str.contains("dailygrind")


**Slicing:** you can treat the column like a Python string.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.str[:5]` slices every value to its first 5 characters.
emails.str[:5]


**Regex — one step at a time.** A *regular expression* is a pattern that describes the shape of
text rather than its exact content. Build it up in three steps.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `re` is Python's built-in regular-expression module -- pattern matching for text.
import re


**Step 1 — a pattern with no groups.** `.` means "any one character" and `+` means "one or more".
So `.+@.+` reads: something, an @ sign, something.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.str.contains` takes a regex, not just plain text. This asks: is there an @ sign
#    with at least one character on each side? A crude but useful "does this look like an email".
emails.str.contains(r".+@.+")


**Step 2 — one group.** Round brackets `( )` mark the part you want to *keep*.
`.str.extract` returns the captured part.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Read the pattern as: "an @ sign, then capture everything after it".
#    That capture is the domain.
emails.str.extract(r"@(.+)")


**Step 3 — three groups.** Same idea, three times, plus two new pieces of notation:

| Piece | Meaning |
|---|---|
| `[A-Z0-9._%+-]` | any one character from this set |
| `\.` | a literal dot (a bare `.` would mean "any character") |
| `+` | one or more of the thing before it |
| `flags=re.IGNORECASE` | treat upper and lower case as the same |


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A regex pattern with three bracketed groups: user, domain, suffix.
#    The `r"..."` prefix stops Python treating backslashes as escape characters.
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"

emails.str.extract(pattern, flags=re.IGNORECASE)


> **Look at Daniel's row.** His address is `daniel.lim@dailygrind.com.sg`, and the pattern put
> `dailygrind.com` in the domain group and `sg` in the suffix. That is not a bug in the regex — it
> is the regex doing exactly what you asked, on data whose shape you had not thought about.
> Every regex you write needs testing against the awkward cases, not the tidy ones.


**Retrieving elements:** chain `.str` calls to get specific parts of a match.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.findall` returns a list of matches per row; `.str[0]` takes the first (and only) one,
#    which is a tuple of the three captured pieces.
matches = emails.str.findall(pattern, flags=re.IGNORECASE).str[0]

matches


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Pull item 1 out of each tuple. Python counts from 0, so 1 is the middle piece: the domain.
matches.str[1]


`extract` is usually the friendlier tool: it puts each group straight into its own column.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Name the groups and they become the column names -- much easier to read six months later.
emails.str.extract(r"(?P<user>[^@]+)@(?P<domain>.+)")


### 3.4: Categorical Data

Converting text columns to the `category` type saves memory and speeds things up. And `pd.cut`
turns continuous numbers into labelled bands, which is what most reports actually want.


**On our dataset:** the owner does not want 360 revenue figures. She wants to know how many shifts
were quiet, normal or busy.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `pd.cut` turns numbers into labelled bands. `bins` are the cut points and `labels` names them.
#    Read the bins as: 0-200, 200-500, 500-1200. bins are cut points, not ranges, so the first bin is 0 ≤ x < 200, the second is 200 ≤ x < 500, etc.
#   “Quiet” = 0-200, “Normal” = 200-500, “Busy” = 500-1200. Anything outside those ranges becomes NaN.
clean["shift_size"] = pd.cut(
    clean["revenue_sgd"],
    bins=[0, 200, 500, 1200],
    labels=["Quiet", "Normal", "Busy"],
)

clean["shift_size"].value_counts()


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `.astype('category')` tells pandas this column has a small set of repeated values.
#    Same data, less memory -- and some operations get faster.
#    category is a special type of text column that is more efficient for repeated values, and it
#    also allows you to specify an order for the categories if you want to do comparisons.
#    The categories are ordered by the order they first appear in the data, not alphabetically.

clean["outlet_name"] = clean["outlet_name"].astype("category")

clean["outlet_name"].dtype


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 One-hot encoding: turn one text column into several 0/1 columns, one per daypart, so text data becomes numerical data.
#    This is what most machine-learning models need instead of text.

pd.get_dummies(clean["daypart"], prefix="is").head()


Drills on smaller examples follow.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A column of repeated drink names. `* 2` repeats the list, giving 8 rows.
drinks = pd.Series(["latte", "kopi", "latte", "latte"] * 2)

drinks


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Only two distinct drinks exist, even though there are 8 rows.
drinks.unique()


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 How many of each. This repetition is exactly what the `category` type optimises.
drinks.value_counts()


**Using the pandas `category` type:**


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Build a realistic little dataset. `rng` is a random-number generator with a fixed seed,
#    so everyone gets the same numbers.
rng = np.random.default_rng(seed=12345)
n = 8
drink_df = pd.DataFrame({
    "drink": ["latte", "kopi", "latte", "latte"] * 2,
    "count": rng.integers(3, 15, size=n),
    "weight": rng.uniform(0, 4, size=n).round(2),
})

drink_df


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.astype('category')` asks pandas to store codes plus a lookup table instead of
#    repeating the text. Notice the dtype in the output.
drink_cat = drink_df["drink"].astype("category")

drink_cat


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Save the converted column back into the table so the change actually sticks.
drink_df["drink"] = drink_cat

drink_df.dtypes


**Binning data (`pd.cut`):** converting continuous numbers into categorical bands.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A plain Python list of shift takings -- continuous numbers we are about to group.
#    `pd.cut` turns numbers into labelled bands. `bins` are the cut points and `labels` names them.
#   if no labels are given, the output is a categorical column with the bin ranges as labels.

takings = [80, 145, 210, 260, 340, 480, 505, 610, 720, 880, 940, 1120]

bins = [0, 200, 500, 1200]

shifts = pd.cut(takings, bins)

shifts


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Which bin each value landed in, as a code number.
shifts.codes


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 The list of bin ranges that were created.
shifts.categories


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Count how many values fell into each bin -- an instant histogram in table form.
pd.Series(shifts).value_counts()


**Binning parameters:**


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `right=False` flips which edge is included: now 200 starts the middle bin
#    rather than ending the first one. Worth checking whenever a value sits exactly on a boundary.
pd.cut(takings, bins, right=False)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Give the bins human-readable names instead of number ranges. The list of labels must
#    have exactly one fewer entry than the list of bin edges.
pd.cut(takings, bins, labels=["Quiet", "Normal", "Busy"])


If you pass an integer instead of edges, pandas computes equal-width bins for you.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Pass a plain number and pandas splits the range into that many equal-width bins.
#    `precision=2` just rounds the printed boundaries.
#   bin edge is exclusive on the left (a, b], so the minmum value would fall outside the first bin, the fix is to extend the range by 0.1% on the lower end, or use `include_lowest=True` to include the minimum value in the first bin.)
# range = 1120 - 80 = 1040, so each bin width = 1040 / 4 = 260
# pad = 1040 * 0.001 = 1.04
# left = 80 - 1.04 = 78.96

pd.cut(np.array(takings), 4, precision=2)


**Indicator / dummy variables:** converting a categorical column into binary columns.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A small table with a text 'key' column, ready for encoding.
keys_df = pd.DataFrame({"key": list("bbacab"), "data1": range(6)})

keys_df


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 'One-hot encoding': one new 0/1 column per distinct value.
# get_dummies() is a convenient way to do this, and it automatically names the new columns after the distinct values in the original column. The result is a DataFrame with the same number of rows as the original, but with additional columns for each unique value in the 'key' column, filled with 0s and 1s indicating the presence of that value in each row.

pd.get_dummies(keys_df["key"])


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `prefix=` puts a label in front, so you can tell where the columns came from
#    after you join them onto something else.

pd.get_dummies(keys_df["key"], prefix="key")


Recipe: combining `get_dummies` with `cut`.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Combine the two ideas: `cut` groups the numbers into bins, then `get_dummies` turns
#    each bin into its own 0/1 column. Common as a final step before modelling.

values = np.random.default_rng(seed=12345).uniform(size=10)
edges = [0, 0.2, 0.4, 0.6, 0.8, 1.0]

pd.get_dummies(pd.cut(values, edges))


### 3.5: Type Conversion and Grouping

Two jobs left. `date_text` is still **text**, so we cannot ask a single date question of it. And we
still have no way to compare outlets — which is the question the owner actually asked.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 Right now the dates are just strings. `.dtype` on one column confirms it: `object`.
clean["date_text"].dtype


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `pd.to_datetime` parses text into real dates. `dayfirst=True` matters: "06/06/2025" is
#    unambiguous but "01/06/2025" is 1 June here and 6 January in the US, and pandas cannot know
#    which you meant. Tell it, or your report can be wrong by months with no error raised.
#.   '<M8[ns]' is the internal type for a date column, which is a 64-bit integer counting nanoseconds since 1970-01-01. The human-readable format is YYYY-MM-DD.

clean["date"] = pd.to_datetime(clean["date_text"], dayfirst=True)

clean["date"].dtype


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `.dt` is the date accessor, the way `.str` is for text. Now these questions are possible:
print("first day:", clean["date"].min().date())
print("last day: ", clean["date"].max().date())

# 👉 Pull parts out of a date to group by later.
clean["weekday"] = clean["date"].dt.day_name()

clean[["date", "weekday"]].head()


**Grouping: split → apply → combine.** `groupby` splits the rows into groups, applies a
calculation to each group, and combines the answers into one table. It is the single most useful
summarising tool in pandas.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 Split by outlet, then total the revenue within each group. Read it as a sentence:
#    "group by outlet name, take the revenue column, add it up".
clean.groupby("outlet_name", observed=True)["revenue_sgd"].sum().round(2)


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 Several statistics at once with `.agg()`. Each line reads:
#    new_column_name = (which column, what calculation).
june_summary = clean.groupby("outlet_name", observed=True).agg(
    revenue=("revenue_sgd", "sum"),
    shifts=("revenue_sgd", "size"),
    avg_shift=("revenue_sgd", "mean"),
    tickets=("tickets", "sum"),
)
june_summary = june_summary.round(2)

june_summary


**This table is the point of the whole lesson.** It is only trustworthy because every number
behind it was checked: the sentinels are gone, the duplicate batch is gone, the mis-keyed
\$98,000 is gone, and the four cafés are four cafés rather than twelve.

Here is the same question asked of the raw file:


In [ ]:
# ===== 🏪 REAL DATASET (`raw`) — our coffee-shop data =====
# 👉 The same summary, on the untouched original. Twelve "outlets", and a total that is
#    more than 50% too high.
raw_numeric = raw.copy()
raw_numeric["revenue_sgd"] = pd.to_numeric(
    raw_numeric["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
)

raw_numeric.groupby("outlet")["revenue_sgd"].sum().round(2)


> **Twelve rows instead of four, and no single row you could put in front of the owner.** This is
> what "the data was messy" costs in practice: not a slightly wrong answer, but a table nobody
> can use.


### 🛠️ Group Exercise 3 — Transformation (10 min)

Split (a), (b), (c) across your group, then compare. Scaffolding fades as you go.

> **(a) Fill in the blanks — mapping** — on `practice3` below, replace *both* bad codes with `NaN`:
> ```python
> practice3.replace([____, ____], np.nan)
> ```
> *Expected:* two holes appear.
>
> **(b) From scratch — strings** — from `clean["email"]`, extract just the part *before* the @ sign
> into a new column called `email_user`. *Hint:* `.str.split("@").str[0]`, or the regex
> `r"([^@]+)@"` with `.str.extract`.
>
> **(c) From scratch — grouping** — build a table with one row per `weekday` showing total revenue
> and the number of shifts, sorted from busiest day to quietest.
> *Hint:* `groupby` → `.agg(...)` → `.sort_values(...)`.
>
> **(d) Explain, no code** — you used `pd.cut` with `bins=[0, 200, 500, 1200]`. What happens to a
> shift that took \$1,350? Look at `clean["shift_size"].isna().sum()` and say why `pd.cut` is
> safer than writing three `if` statements.


**(a)** uses the `practice3` table below.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Practice data for the exercise: -999 and 999 stand in for two different bad codes.
practice3 = pd.Series([120.0, -999.0, 340.0, 999.0, 210.0])

practice3


> **Check yourself:** `practice3.replace(...)` returns a *new* Series. Nothing changes in
> `practice3` unless you assign the result back to it.


---

## Part 4: Reading and Writing Data (Self-Study)

**Learning outcome 4:** *Read and write data across multiple file formats (CSV, JSON, Excel,
databases).*

**Goal:** get the cleaned table out of this notebook and into a file somebody else can use.
Nothing in Parts 1–3 needed this, which is why it comes last.

⏱️ ~15 min


### 4.1: Reading Data

Pandas is flexible about input formats. Start with a plain comma-separated file.


In [ ]:
# 👉 The leading `!` runs a terminal command rather than Python. `cat` prints a file as-is,
#    which is the fastest way to see what you are about to load.
!cat ../data/ex1.csv


In [ ]:
# 👉 `read_csv` turns that text file into a DataFrame. The path `../data/` means
#    'go up one folder from notebooks/, then into data/'.
pd.read_csv("../data/ex1.csv")


**Scenario:** what if the file has no header row? Without telling pandas, it will use your first
row of real data as the column names.


In [ ]:
# 👉 Same peek, at a file whose first line is real data rather than column names.
!cat ../data/ex2.csv


In [ ]:
# 👉 `header=None` says 'there are no column names', so pandas numbers them 0, 1, 2...
pd.read_csv("../data/ex2.csv", header=None)


In [ ]:
# 👉 Better: supply your own column names with `names=`.
pd.read_csv(
    "../data/ex2.csv",
    names=["date", "outlet_id", "daypart", "tickets", "items", "revenue_sgd"],
)


**Indexing:** you can promote a column to be the row labels.


In [ ]:
# 👉 `index_col=` promotes one column to be the row labels instead of an ordinary column.
names = ["date", "outlet_id", "daypart", "tickets", "items", "revenue_sgd"]

pd.read_csv("../data/ex2.csv", names=names, index_col="outlet_id")


**Junk before the header:** exports often start with a couple of comment lines.


In [ ]:
# 👉 Two lines of preamble before the real header.
!cat ../data/ex4.csv


In [ ]:
# 👉 `skiprows=` drops lines by position. `comment="#"` is the more robust version:
#    it ignores any line starting with that character, wherever it appears.
pd.read_csv("../data/ex4.csv", comment="#")


**Handling missing values at load time:** pandas already recognises empty fields, `NA`, `NULL` and
`n/a`. It cannot guess *your* system's conventions.


In [ ]:
# 👉 This file writes missing values in several different ways.
!cat ../data/ex5.csv


In [ ]:
# 👉 Out of the box, pandas recognises the empty field, 'NA', 'NULL' and 'n/a' -- but -999
#    comes through as a number, because there is no way for pandas to know it is a code.
result = pd.read_csv("../data/ex5.csv")

result


In [ ]:
# 👉 `na_values=` adds your own conventions to the list. Doing it here, at load time, is
#    tidier than fixing it later -- and it means the sentinel never poisons a statistic.
pd.read_csv("../data/ex5.csv", na_values=["-999"])


In [ ]:
# 👉 Sometimes 'missing' is spelled differently per column. A dictionary says which
#    text counts as missing in which column.
pd.read_csv("../data/ex5.csv", na_values={"tickets": ["-999"], "revenue_sgd": ["n/a"]})


**Reading Excel:** a workbook can hold several sheets, so you usually look before you load.


In [ ]:
# 👉 Open the workbook and see what is inside it.
workbook = pd.ExcelFile("../data/cafe_june_workbook.xlsx")

workbook.sheet_names


In [ ]:
# 👉 `.parse()` reads one named sheet into a DataFrame.
workbook.parse(sheet_name="Outlets")


In [ ]:
# 👉 The one-line shortcut when you already know the sheet you want.
pd.read_excel("../data/cafe_june_workbook.xlsx", sheet_name="June").head()


In [ ]:
# 👉 Now compare. This workbook (and the database in 4.3) holds the *reference* clean June --
#    the version with the eight damaged shifts restored from the original till logs.
reference = pd.read_excel("../data/cafe_june_workbook.xlsx", sheet_name="June")

print(f"your cleaned June:      ${clean['revenue_sgd'].sum():>11,.2f}")
print(f"reference clean June:   ${reference['revenue_sgd'].sum():>11,.2f}")
print(f"difference:             ${reference['revenue_sgd'].sum() - clean['revenue_sgd'].sum():>11,.2f}")


> **Your number is different, and that is the right answer.** Eight shifts in the raw export had lost
> their true figure — four sentinels, one mis-key, three blanks — and you filled them with the median,
> because the median is the best defensible guess available *from that file*. The reference version was
> reconstructed from the original till logs, which you were never given.
>
> \$916 on \$175,669 is **0.5%**, and you can name every dollar of it. That is what a clean dataset
> looks like in practice: not identical to the truth, but different from it in ways you can explain.
>
> This reference file is the June slice of what Lesson 1.9 opens next.


### 4.2: Writing Data (Exporting)

Saving is the mirror of reading.


In [ ]:
# 👉 `to_csv` writes the DataFrame back out to a file.
result.to_csv("../data/out.csv")

# 👉 Peek at what was written. Notice the extra unnamed first column -- that is the index.
!cat ../data/out.csv


**Tip:** you usually want `index=False`, so the row numbers are not saved as a mystery column.


In [ ]:
# 👉 `index=False` leaves the row labels out, which is what you want when the index is
#    just 0, 1, 2... and carries no meaning.
result.to_csv("../data/out.csv", index=False)

!cat ../data/out.csv


**JSON export:** the format APIs and web services speak.


In [ ]:
# 👉 `orient="records"` writes a list of one object per row, which is what most web APIs expect.
result.to_json("../data/out.json", orient="records")

!cat ../data/out.json


In [ ]:
# 👉 And straight back in.
pd.read_json("../data/out.json", orient="records")


**Save our session's work.** Everything from Parts 2 and 3 lives in `clean`. Write out the columns
a colleague would actually want.


In [ ]:
# 👉 Keep the useful columns, in a sensible order, with the tidy names.
final = clean[[
    "date", "outlet_name", "daypart", "tickets", "items", "revenue_sgd", "staff", "manager",
]].sort_values(["date", "outlet_name", "daypart"])

final.to_csv("../data/cafe_june_clean.csv", index=False)

final.head()


In [ ]:
# 👉 Beat 4 of the 1.8 habit -- verify. Read back what you just wrote and check it matches.
check = pd.read_csv("../data/cafe_june_clean.csv")

print("rows written:", len(check))
print(f"June revenue: ${check['revenue_sgd'].sum():,.2f}")
print("date column came back as:", check["date"].dtype)


> **Note that last line.** The date went out as a real date and came back as **text**. CSV has no
> way to store types, so every reader has to re-parse them (`parse_dates=["date"]`). Databases and
> pickle files remember types; CSV and JSON forget them. That is worth knowing before you build a
> pipeline out of CSVs.


**Excel export:**


In [ ]:
# 👉 Writing Excel needs a 'writer' object when you want control over sheets.
#    Using `with` closes the file for you, which is what actually saves it.
with pd.ExcelWriter("../data/cafe_june_clean.xlsx") as writer:
    final.to_excel(writer, sheet_name="June", index=False)
    june_summary.to_excel(writer, sheet_name="Summary")

pd.ExcelFile("../data/cafe_june_clean.xlsx").sheet_names


### 4.3: Databases

Most real data lives in a database rather than a file. `sqlalchemy` is the library pandas uses to
talk to them, and the pandas side barely changes.


In [ ]:
# 👉 SQLAlchemy speaks to many database engines through one interface.
import sqlalchemy as sqla

# 👉 A connection string says which engine and which file. `sqlite:///` is a local file --
#    no server to install, which is why it is the right thing for a lesson.
engine = sqla.create_engine("sqlite:///../data/cafe.db")

# 👉 What tables are in there? `inspect` is the version-safe way to ask.
sqla.inspect(engine).get_table_names()


In [ ]:
# 👉 Give a table name and pandas reads the whole table into a DataFrame.
pd.read_sql_table("outlets", engine)


In [ ]:
# 👉 Or hand it real SQL and only the query result comes back. Same DataFrame either way.
#    Everything you learned about SQL in Lessons 1.3-1.5 works here.
query = (
    'SELECT outlet_id, SUM(revenue_sgd) AS revenue, COUNT(*) AS shifts '
    'FROM daily_sales_june '
    'GROUP BY outlet_id '
    'ORDER BY revenue DESC'
)

pd.read_sql(query, engine)


In [ ]:
# 👉 Writing back: `to_sql` creates (or replaces) a table from a DataFrame.
#    This is how a cleaned table gets handed to the rest of the business.
final.to_sql("june_clean", engine, index=False, if_exists="replace")

sqla.inspect(engine).get_table_names()


> **Task:** write a *filtered* table to the database.
> 1. Filter `final` to Marina Bay only.
> 2. Write it to a new table called `marina_june`.
> 3. Read it back with `pd.read_sql` and check the row count is 90 (30 days × 3 dayparts).


> **Final challenge:**
> 1. Read `daily_sales_june` back out of the database.
> 2. Compute total revenue per daypart with SQL (`GROUP BY daypart`) *and* with pandas
>    (`groupby("daypart")`).
> 3. Confirm the two agree. Two tools, one answer — that is the check worth building the habit on.


## 🎯 Wrap-Up

1. **Always run the five-move first look** on a new file — head, shape, info, dtypes, describe.
   It takes two minutes and it found all five of today's problems.
2. **Wrong types hide everything.** The impossible \$98,000 and the -999 sentinels were invisible
   while revenue was text. Move 4 (`.dtypes`) is not paperwork.
3. **Cleaning decisions depend on what the value means**, not on what is convenient: fill, drop,
   cap or mark missing are four different answers to four different situations.
4. **Order of operations matters.** Remove the sentinels *before* you compute the median you are
   going to impute with.
5. **Standardise your keys before you trust a check.** The duplicate check in 2.2 was meaningless
   until the twelve outlet spellings became four.
6. **A clean table is not the goal; a trustworthy answer is.** Section 3.5 — four rows, one per
   café — is what all the cleaning was for.

**Next Steps:**
- Complete the [Assignment](./assignment.md) — audit a second month of the same export.
- Next lesson: **1.9 EDA Advanced** opens `daily_sales.csv` — the same export for **all 18 months**,
  cleaned exactly the way you just cleaned June, and asks what the pattern is.

> **The handoff, in one number.** Your cleaned June total is **\$174,753**. In Lesson 1.9's clean
> 18-month file, June 2025 is **\$175,669** — a difference of **\$916**, or 0.5%, and you can say
> exactly where it comes from: **eight shifts** (four sentinels, one mis-keyed figure and three blanks)
> where you put the median in place of a number this export had destroyed.
>
> That is the difference between clean data and lucky data: not that your figure matches, but that
> you can account for why it does not.


---

## 📎 Appendix — Self-Study

These topics fall outside today's four learning outcomes, but come up constantly in practice.
Deep dives on categorical internals, awkward CSV parsing and pickle files live in `reference.md`.


### Permutation and Random Sampling

Reordering rows, or taking a random subset — the basis of train/test splits later in the course.


In [ ]:
# 👉 A 5x7 table of the numbers 0-34 to make row shuffling easy to see.
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))

df


In [ ]:
# 👉 `permutation(5)` returns the numbers 0-4 in random order -- a shuffled seating plan.
sampler = np.random.default_rng(seed=12345).permutation(5)

sampler


In [ ]:
# 👉 `.iloc[...]` selects rows *by position*, so the table comes back in the shuffled order.
df.iloc[sampler]


In [ ]:
# 👉 `.take()` does the same job and reads a little more clearly.
df.take(sampler)


Permuting columns:


In [ ]:
# 👉 `df.shape[1]` is the number of columns, so this shuffles the column positions.
column_sampler = np.random.default_rng(seed=12345).permutation(df.shape[1])

column_sampler


In [ ]:
# 👉 `axis=1` applies the shuffle to columns instead of rows.
df.take(column_sampler, axis=1)


**Random sample:** getting a random subset without the manual shuffle.


In [ ]:
# 👉 `.sample()` skips the manual shuffle: just ask for 3 random rows.
df.sample(n=3)


In [ ]:
# 👉 `replace=True` puts each pick back in the hat, so 10 draws from 5 rows is possible
#    and rows can repeat. This is 'sampling with replacement'.
df.sample(n=10, replace=True)


> **Task:** sample `df` using the parameter `frac` (a fraction of the rows) instead of `n` (a count).


---

## ✅ Sample Solutions — Group Exercises 1, 2 & 3

One worked answer per task. These are *a* solution, not *the* solution — if your code gives the
same answer by another route, it is right. Try each exercise before reading on.


### Group Exercise 1 — Summarising

**(b) Sort the outlet counts by frequency, smallest first.**


In [ ]:
# ===== 🏪 REAL DATASET (`raw`) — our coffee-shop data =====
# 👉 `.value_counts()` already sorts biggest-first, so we re-sort the *values* of that
#    count table ascending to put the rarest spelling on top.
raw["outlet"].value_counts().sort_values(ascending=True)


Twelve rows, smallest count on top. Read it as a finding, not a list: no single spelling
dominates — `Holland V` (47) is used *more* than the tidy `Holland Village` (44), and Raffles Place
is split four ways across 23 rows each. Group by `outlet` today and every café is understated.

**(c) Which shift sold the most items?**


In [ ]:
# ===== 🏪 REAL DATASET (`raw`) — our coffee-shop data =====
# 👉 `.idxmax()` gives the *row label* of the largest value (not the value itself),
#    then `.loc[...]` pulls that whole row out so we can see which shift it was.
busiest = raw["items"].idxmax()

raw.loc[busiest]


In [ ]:
# 👉 Same answer as a one-row table rather than a Series: pass a *list* of labels.
raw.loc[[raw["items"].idxmax()]]


**(d) `info()` says 364 non-null tickets but the table has 366 rows — where did 2 go?**

Two shifts have no ticket count recorded at all: the cell is empty in the export, so pandas
stores `NaN` and `.info()` does not count it as non-null. Nothing was deleted — the till simply
never wrote a number for those two shifts.

Showing them as missing is better than showing `0` because **`0` is a claim and `NaN` is an
admission**. `0` says "this shift served zero customers" — a real, meaningful business fact.
`NaN` says "we do not know what this shift served". Writing `0` would silently drag the mean
tickets-per-shift down, make the shift look like a genuine closure, and hide a data-collection
problem you may want to go and fix. Pandas also skips `NaN` in `.mean()`, `.sum()` and friends
automatically, so leaving it missing keeps the summary statistics honest.


### Group Exercise 2 — Data Quality

**(b) Drop only the rows with fewer than 2 real values.**


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `thresh=2` means "keep a row if it has at least 2 non-missing values".
#    Row 4 (all three values missing) is the only one that fails the test.
practice.dropna(thresh=2)


5 of the 6 rows survive. Note the wording: `thresh` is a *keep* threshold, not a drop
threshold — rows 0 and 2 each have one hole but still hold 2 real values, so they stay.

**(c) Fill the holes in column `b` with that column's median, leaving `a` and `c` untouched.**


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A dictionary tells `.fillna()` which column gets which filler, so only `b` is touched.
#    `.median()` skips the NaNs itself, so it is the median of the real values.
practice.fillna({"b": practice["b"].median()})


In [ ]:
# 👉 Verify: column `b` should have no holes left, while `a` and `c` keep the hole in row 4.
practice.fillna({"b": practice["b"].median()}).isna().sum()


**(d) How many rows of `raw` had a `revenue_raw` that could not be read as a number at all?**


In [ ]:
# ===== 🏪 REAL DATASET (`raw`) — our coffee-shop data =====
# 👉 Re-run the same conversion on the *untouched* raw column and count what fell through.
#    `errors="coerce"` puts NaN wherever the text was not readable as a number.
unreadable = pd.to_numeric(
    raw["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
).isna().sum()

print("rows that could not be read as a number:", unreadable)


In [ ]:
# 👉 Look at the actual offenders, so the number is not just a number.
bad = pd.to_numeric(
    raw["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
).isna()

raw.loc[bad, ["outlet", "daypart", "revenue_raw"]]


**Why `errors="coerce"` is safer than letting the conversion raise:** raising stops the whole
conversion on the *first* bad value, so you fix one row, re-run, hit the next bad row, and repeat —
and you never see how big the problem is. `coerce` converts everything it can, parks every failure
as `NaN`, and lets you count and inspect all of them in one pass. You still have to decide what to
do with them; you just get to see the full damage first.


### Group Exercise 3 — Transformation

**(a) Replace both bad codes with `NaN`.**


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.replace()` takes a list of values to swap out and one value to swap in.
#    -999 and 999 are two different sentinels; both mean "no reading".
practice3.replace([-999.0, 999.0], np.nan)


In [ ]:
# 👉 Remember: that returned a *new* Series. `practice3` itself is unchanged
#    until you assign the result back.
practice3


**(b) Extract the part before the @ sign into `email_user`.**


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `.str.split("@")` gives a list per row, e.g. ["aisha.rahman", "dailygrind.sg"];
#    `.str[0]` then takes the first piece of each list.
clean["email_user"] = clean["email"].str.split("@").str[0]

clean[["email", "email_user"]].head()


In [ ]:
# 👉 Same answer with a regex: "one or more characters that are not @, followed by @".
#    The brackets mark the part we want kept. `.str.extract` returns a DataFrame,
#    so `[0]` pulls out that single column.
clean["email"].str.extract(r"([^@]+)@")[0].head()


**(c) Total revenue and number of shifts per weekday, busiest day first.**


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 One row per weekday, two numbers per row: the money total and how many shifts
#    went into it. Naming the outputs in `.agg()` saves renaming afterwards.
by_weekday = (
    clean.groupby("weekday")
    .agg(total_revenue=("revenue_sgd", "sum"), shifts=("revenue_sgd", "size"))
    .sort_values("total_revenue", ascending=False)
)

by_weekday


Always read the `shifts` column next to the money. A day with fewer shifts recorded will
look quieter even if its *per-shift* takings are the same — the count is what stops you drawing
that wrong conclusion.


**(d) What happens to a shift that took \$1,350 under `pd.cut(bins=[0, 200, 500, 1200])`?**


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 Anything outside the outermost edges becomes NaN. Count them, then look at them.
print("shifts with no size label:", clean["shift_size"].isna().sum())

clean.loc[clean["shift_size"].isna(), ["outlet_name", "daypart", "revenue_sgd", "shift_size"]]


\$1,350 is above the top edge of 1200, so it falls into **no bin at all** and `pd.cut` labels it
`NaN`. The value is not silently dropped and not silently squeezed into the top bucket — it is
marked as unclassified, and `.isna().sum()` will show it to you.

**Why that is safer than three `if` statements.** A hand-written chain like

```python
if r < 200:   size = "quiet"
elif r < 500: size = "normal"
else:         size = "busy"
```

has a silent `else` that swallows *everything* above 500 — \$1,350, \$98,000 and a mis-keyed
\$980,000 all come out labelled "busy", and nothing tells you. It also quietly mislabels missing
revenue and negative sentinels. `pd.cut` makes the boundaries explicit data (`bins=[...]`), applies
them to the whole column in one vectorised pass, refuses to guess about anything outside the range,
and gives you one honest counter — `.isna().sum()` — for how often it had to refuse.
